# 01 — Perception: the ruler (CV) and the eye (one VLM call)

**End to end:** each problem's 12 panels become a written record the agents reason over.
- **Ruler — CV, free:** exact size, position, long-axis direction, elongation, convexity, corners, corner angles
  (acute / right), blob count, box-level inside / overlap.
- **Eye — one VLM call (gpt-4o):** the 12 panels are shuffled into one labelled grid, so the model cannot tell left
  from right. It fills one checklist for every panel at four levels: parts, objects, relations and the whole panel,
  plus sameness and symmetry. The checklist and why it has these fields: `docs/eye_perception_rules.txt`.
  It describes; it does not solve.
- **What changes across the 12 panels:** computed by code from the eye and the ruler. A property that is the
  same in all 12 panels cannot be the rule.

No field comes from both sources, so the record never contradicts itself. Vision is used **once per problem**;
everything after this notebook is text. Output: `results/perception/<problem>__<model>__fields-v4.json`.

In [ ]:
%run ./00_common.ipynb

In [ ]:
PROBLEMS = PROBLEM_SETS[os.environ.get("PROBLEM_SET", "SMOKE")]
EXAMPLE = "bp0038" if "bp0038" in PROBLEMS else PROBLEMS[0]
print(len(PROBLEMS), "problems; example:", EXAMPLE)
show_problem(EXAMPLE)

### What the eye sees: the shuffled grid (no left/right cue)

In [ ]:
_, _, grid = shuffled_grid(EXAMPLE, seed=zlib.crc32(f"{EXAMPLE}-fields".encode()))
plt.figure(figsize=(8, 6.5)); plt.imshow(grid, cmap="gray"); plt.axis("off")
plt.title(f"{EXAMPLE}: the single image the vision model sees"); plt.show()

### The record per panel: eye + ruler, one source per fact

In [ ]:
METER.tag = "perception: 1 VLM call per problem"
fields, cv = vlm_fields(EXAMPLE), load_cv(EXAMPLE)
pd.set_option("display.max_colwidth", 150)
display(pd.DataFrame([{"panel": pid, "EYE (VLM)": eye_text(fields[pid]), "RULER (CV)": ruler_text(cv[pid])} for pid in PIDS]))
words = panel_records(EXAMPLE, "hybrid", words=True)
print("the same panel as the judge reads it (ruler in words):\n ", words["left_1"])
print("\n" + changes_text(EXAMPLE))

### Run perception for every problem in the set (cached: re-running costs nothing)

In [ ]:
for bp in PROBLEMS:
    f = vlm_fields(bp)
    filled = sum(1 for pid in PIDS if isinstance(f.get(pid), dict) and f[pid])
    print(f"{bp}: {filled}/12 panels read   (${perception_cost(bp):.4f})")
METER.report()